# CrossHL-VLM Showcase

Notebook interface for inspecting HSI--LiDAR datasets, running CrossHL-VLM ablations, and generating analysis plots.

Core behavior:
- Cross-HL remains the classifier.
- CLIP is frozen and provides text prototypes only for semantic regularization.
- HSI and LiDAR are processed in their native modalities.
- No RGB conversion is used.


## Required Imports


In [ ]:
from pathlib import Path
import csv
import json
import os
import random
import sys
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Subset

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data import HSILidarDataset, apply_spectral_perturbation, make_fewshot_subset, make_kshot_subset
from model.CrossHL_model import CrossHL_Transformer
from prompts import PROMPT_SETS, build_text_prototypes, get_class_names, prototype_similarity_stats

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Device:", DEVICE)


## Helper Functions


In [ ]:

AVAILABLE_DATASETS = ["Trento", "Houston", "MUUFL"]


def set_seed(seed=14):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def pct_tag(pct):
    return f"pct{int(round(float(pct) * 100))}"


def split_tag(split_kind, split_value):
    return f"k{int(split_value)}" if split_kind == "shot" else pct_tag(split_value)


def split_label(split_kind, split_value):
    return f"{int(split_value)}-shot" if split_kind == "shot" else f"{int(round(float(split_value) * 100))}%"


def split_sort_key(label):
    label = str(label)
    if label.endswith("-shot"):
        return (0, int(label.replace("-shot", "")))
    if label.endswith("%"):
        return (1, float(label.replace("%", "")))
    return (2, label)


def load_dataset_context(dataset):
    global DATASET, train_full, test_full, class_names, NC, NCLIDAR, CLASSES, PATCH_SIZE
    DATASET = dataset
    train_full = HSILidarDataset(PROJECT_ROOT, dataset=DATASET, split="train")
    test_full = HSILidarDataset(PROJECT_ROOT, dataset=DATASET, split="test")
    class_names = get_class_names(DATASET)
    NC = train_full.hs_image.shape[1]
    NCLIDAR = train_full.lidar_image.shape[1]
    CLASSES = len(torch.unique(train_full.lbls))
    PATCH_SIZE = train_full.hs_image.shape[-1]
    if CLASSES != len(class_names):
        raise ValueError(f"{DATASET}: labels contain {CLASSES} classes but prompts define {len(class_names)} names.")
    return train_full, test_full, class_names


def dataset_inventory(datasets=AVAILABLE_DATASETS):
    rows = []
    for dataset in datasets:
        train_ds = HSILidarDataset(PROJECT_ROOT, dataset=dataset, split="train")
        test_ds = HSILidarDataset(PROJECT_ROOT, dataset=dataset, split="test")
        labels = train_ds.lbls.detach().cpu().numpy()
        vals, counts = np.unique(labels, return_counts=True)
        rows.append({
            "dataset": dataset,
            "classes": len(vals),
            "train_samples": len(train_ds),
            "test_samples": len(test_ds),
            "hsi_bands": train_ds.hs_image.shape[1],
            "lidar_channels": train_ds.lidar_image.shape[1],
            "patch": train_ds.hs_image.shape[-1],
            "min_train_per_class": int(counts.min()),
            "max_train_per_class": int(counts.max()),
        })
    return pd.DataFrame(rows)


def make_run_dir(base="runs", name=None):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = name or f"notebook_{DATASET}_{timestamp}"
    run_dir = PROJECT_ROOT / base / run_name
    (run_dir / "checkpoints" / DATASET).mkdir(parents=True, exist_ok=True)
    (run_dir / "logs" / DATASET).mkdir(parents=True, exist_ok=True)
    (run_dir / "analysis").mkdir(parents=True, exist_ok=True)
    return run_dir


def append_csv_row(path, row):
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def write_csv(path, rows):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def make_balanced_eval_subset(dataset, max_samples=1024, seed=14):
    labels = dataset.lbls.detach().cpu()
    classes = torch.unique(labels).tolist()
    per_class = max(1, int(max_samples) // max(1, len(classes)))
    generator = torch.Generator().manual_seed(seed)
    selected = []
    counts = {}
    all_indices = torch.arange(len(labels))
    for class_id in classes:
        class_indices = all_indices[labels == class_id]
        n_keep = min(per_class, len(class_indices))
        perm = class_indices[torch.randperm(len(class_indices), generator=generator)]
        selected.append(perm[:n_keep])
        counts[int(class_id)] = int(n_keep)
    return Subset(dataset, torch.cat(selected).tolist()), counts


# Dataset Overview

Inspect available HSI--LiDAR datasets and class balance before launching a run.


In [ ]:

DATASET = "Trento"
FM = 16
BATCH_SIZE = 64
TEST_BATCH_SIZE = 500
SEED = 14
SPLIT_SEED = 14

set_seed(SEED)
load_dataset_context(DATASET)

print("Dataset:", DATASET)
print("Train samples:", len(train_full))
print("Test samples:", len(test_full))
print("HSI:", tuple(train_full.hs_image.shape))
print("LiDAR:", tuple(train_full.lidar_image.shape))
print("Classes:", CLASSES, class_names)

display(dataset_inventory())


In [ ]:

SPLIT_KIND = "shot"  # "shot" or "pct"
INSPECT_SHOT = 20
INSPECT_PCT = 0.10

if SPLIT_KIND == "shot":
    inspect_subset, inspect_counts = make_kshot_subset(train_full, shots=INSPECT_SHOT, seed=SPLIT_SEED)
    print(f"Few-shot setting: {INSPECT_SHOT}-shot per class")
else:
    inspect_subset, inspect_counts = make_fewshot_subset(train_full, pct=INSPECT_PCT, seed=SPLIT_SEED)
    print(f"Few-shot setting: {int(INSPECT_PCT * 100)}%")

print("Subset samples:", len(inspect_subset))
print("Per-class selected samples:", {class_names[k]: v for k, v in inspect_counts.items()})


# Cross-HL Model

The classifier path is the same Cross-HL logic: fused CLS feature goes to `fclayer`. The VLM branch is optional and starts after the encoder feature.


In [ ]:
model = CrossHL_Transformer(
    FM=FM,
    NC=NC,
    NCLidar=NCLIDAR,
    Classes=CLASSES,
    patchsize=PATCH_SIZE,
).to(DEVICE)
model.eval()

sample_loader = DataLoader(inspect_subset, batch_size=4, shuffle=False)
hsi, lidar, labels = next(iter(sample_loader))
hsi = hsi.to(DEVICE)
lidar = lidar.to(DEVICE)
labels = labels.to(DEVICE)

with torch.no_grad():
    logits_only = model(hsi, lidar)
    logits_branch, img_embed, cls_features = model(
        hsi,
        lidar,
        return_embed=True,
        return_features=True,
    )

print("Logits only:", tuple(logits_only.shape))
print("Logits with branch:", tuple(logits_branch.shape))
print("VLM projection embedding:", tuple(img_embed.shape))
print("Raw fused CLS feature:", tuple(cls_features.shape))
print("Classifier output unchanged when branch is requested:", torch.allclose(logits_only, logits_branch, atol=1e-6))
print(model)


# Prompt Sets

Minimum ablation:
- baseline: no semantic loss
- name: class-name prompts
- spectral: physics/spectral attribute prompts

Extra multimodal ablation:
- spectral_lidar: spectral/material plus LiDAR structure/elevation cues, because Cross-HL aligns a fused HSI+LiDAR feature.


In [ ]:
for mode in ["name", "spectral", "spectral_lidar"]:
    print("\n" + "=" * 100)
    print(mode.upper())
    for class_name, prompts in PROMPT_SETS[DATASET][mode].items():
        print(f"\n{class_name}")
        for prompt in prompts:
            print("  -", prompt)


# CLIP Text Encoder

Set `LOAD_CLIP=True` when running semantic experiments inside the notebook. CLIP is frozen. It only creates text prototypes.


In [ ]:

LOAD_CLIP = True
CLIP_MODEL = "ViT-B-32"
CLIP_PRETRAINED = "laion2b_s34b_b79k"
CENTER_PROTOTYPES = True

clip_model = None
tokenizer = None
text_prototype_cache = {}
text_prototypes = {}


def ensure_clip_loaded():
    global clip_model, tokenizer
    if clip_model is not None and tokenizer is not None:
        return
    if not LOAD_CLIP:
        raise RuntimeError("LOAD_CLIP=False. Set LOAD_CLIP=True before semantic ablations.")
    import open_clip
    clip_model, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAINED)
    tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
    clip_model = clip_model.to(DEVICE).eval()
    for param in clip_model.parameters():
        param.requires_grad = False


def maybe_center_prototypes(prototypes):
    centered = prototypes - prototypes.mean(dim=0, keepdim=True)
    return F.normalize(centered, dim=-1)


def ensure_text_prototypes(dataset):
    global text_prototypes
    if dataset in text_prototype_cache:
        text_prototypes = text_prototype_cache[dataset]
        return text_prototypes
    ensure_clip_loaded()
    prototypes_by_mode = {}
    print(f"Building frozen CLIP text prototypes for {dataset}")
    for mode in ["name", "spectral", "spectral_lidar"]:
        prototypes = build_text_prototypes(clip_model, tokenizer, dataset, mode, DEVICE)
        stats = prototype_similarity_stats(prototypes)
        if CENTER_PROTOTYPES:
            prototypes = maybe_center_prototypes(prototypes)
            centered_stats = prototype_similarity_stats(prototypes)
            print(
                f"{dataset} / {mode}: raw mean/max={stats['mean_offdiag']:.3f}/{stats['max_offdiag']:.3f}; "
                f"centered mean/max={centered_stats['mean_offdiag']:.3f}/{centered_stats['max_offdiag']:.3f}"
            )
        else:
            print(f"{dataset} / {mode}: mean off-diagonal={stats['mean_offdiag']:.3f}, max={stats['max_offdiag']:.3f}")
        prototypes_by_mode[mode] = prototypes
    text_prototype_cache[dataset] = prototypes_by_mode
    text_prototypes = prototypes_by_mode
    return prototypes_by_mode


if LOAD_CLIP:
    ensure_text_prototypes(DATASET)
else:
    print("CLIP not loaded. Baseline and no-CLIP sanity cells can still run.")
    print("Set LOAD_CLIP=True before semantic ablation cells.")


# Evaluation Function


In [ ]:

def evaluate_model(model, loader, device=DEVICE, class_count=None):
    if class_count is None:
        class_count = CLASSES
    y_true = []
    y_pred = []
    model.eval()
    with torch.no_grad():
        for hsi_batch, lidar_batch, label_batch in loader:
            hsi_batch = hsi_batch.to(device)
            lidar_batch = lidar_batch.to(device)
            logits = model(hsi_batch, lidar_batch)
            pred = torch.argmax(logits, dim=1)
            y_true.append(label_batch.cpu().numpy())
            y_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    conf = confusion_matrix(y_true, y_pred, labels=list(range(class_count)))
    class_totals = conf.sum(axis=1)
    per_class = np.full(class_count, np.nan, dtype=np.float64)
    valid_classes = class_totals > 0
    per_class[valid_classes] = (np.diag(conf)[valid_classes] / class_totals[valid_classes]) * 100.0
    aa = float(np.nanmean(per_class)) if np.any(valid_classes) else 0.0
    per_class = np.nan_to_num(per_class, nan=0.0)
    return {
        "oa": accuracy_score(y_true, y_pred) * 100.0,
        "aa": aa,
        "kappa": cohen_kappa_score(y_true, y_pred) * 100.0,
        "per_class": per_class,
        "confusion": conf,
    }


# Training Function

Train one experiment for one split and iteration.


In [ ]:
def train_one_experiment(
    exp_name,
    prompt_mode,
    lambda_sem,
    train_loader,
    test_loader,
    run_dir,
    split_kind,
    split_value,
    iteration,
    epochs=200,
    lr=5e-4,
    weight_decay=5e-3,
    semantic_start_epoch=0,
    lambda_warmup_epochs=50,
    temperature=0.07,
    spectral_noise_std=0.0,
    spectral_gain_std=0.0,
    grad_clip=1.0,
    freeze_pct_threshold=0.0,
    save_best_by_test=False,
    eval_interval=50,
):
    set_seed(SEED + iteration)
    model = CrossHL_Transformer(
        FM=FM,
        NC=NC,
        NCLidar=NCLIDAR,
        Classes=CLASSES,
        patchsize=PATCH_SIZE,
    ).to(DEVICE)

    freeze_early = freeze_pct_threshold > 0 and split_kind == "pct" and split_value <= freeze_pct_threshold
    if freeze_early:
        model.freeze_early_layers()
        print("Early convolution layers frozen for 1% regime.")

    prototypes = None
    if prompt_mode is not None:
        if prompt_mode not in text_prototypes:
            raise RuntimeError(f"Text prototypes for {DATASET}/{prompt_mode} are missing. Set LOAD_CLIP=True and run the CLIP cell.")
        prototypes = text_prototypes[prompt_mode].to(DEVICE)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    ce_loss = nn.CrossEntropyLoss()

    tag = f"{exp_name}_{split_tag(split_kind, split_value)}_lam{lambda_sem:g}"
    best_state = None
    best_eval = None
    best_oa = -1.0
    epoch_rows = []
    start = time.time()

    for epoch in range(epochs):
        model.train()
        if epoch < semantic_start_epoch:
            lambda_scale = 0.0
        else:
            warmup_epoch = epoch - semantic_start_epoch + 1
            lambda_scale = min(1.0, float(warmup_epoch) / max(1, lambda_warmup_epochs))
        current_lambda = lambda_sem * lambda_scale
        ce_total = 0.0
        sem_total = 0.0
        total_loss = 0.0
        batches = 0

        for hsi_batch, lidar_batch, label_batch in train_loader:
            hsi_batch = hsi_batch.to(DEVICE)
            lidar_batch = lidar_batch.to(DEVICE)
            label_batch = label_batch.to(DEVICE)
            hsi_batch = apply_spectral_perturbation(hsi_batch, spectral_noise_std, spectral_gain_std)

            optimizer.zero_grad()
            if prototypes is None or current_lambda <= 0:
                logits = model(hsi_batch, lidar_batch)
                loss_sem = torch.tensor(0.0, device=DEVICE)
            else:
                logits, img_embed = model(hsi_batch, lidar_batch, return_embed=True)
                semantic_logits = (img_embed @ prototypes.T) / temperature
                loss_sem = F.cross_entropy(semantic_logits, label_batch)

            loss_ce = ce_loss(logits, label_batch)
            loss = loss_ce + current_lambda * loss_sem
            loss.backward()
            if grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            ce_total += loss_ce.item()
            sem_total += loss_sem.item()
            total_loss += loss.item()
            batches += 1

        scheduler.step()

        eval_result = None
        if (epoch + 1) % eval_interval == 0 or epoch == epochs - 1:
            eval_result = evaluate_model(model, test_loader)
            if save_best_by_test and eval_result["oa"] > best_oa:
                best_oa = eval_result["oa"]
                best_eval = eval_result
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(
                f"{tag} | iter {iteration} | epoch {epoch + 1}/{epochs} | "
                f"CE={ce_total / max(1, batches):.4f} | Sem={sem_total / max(1, batches):.4f} | "
                f"OA={eval_result['oa']:.2f}"
            )

        epoch_rows.append({
            "epoch": epoch + 1,
            "experiment": exp_name,
            "tag": tag,
            "pct": split_value if split_kind == "pct" else "",
            "shots": split_value if split_kind == "shot" else "",
            "split_label": split_label(split_kind, split_value),
            "iteration": iteration,
            "lambda_sem": current_lambda,
            "loss_ce": ce_total / max(1, batches),
            "loss_sem": sem_total / max(1, batches),
            "loss_total": total_loss / max(1, batches),
            "test_oa": "" if eval_result is None else eval_result["oa"],
            "test_aa": "" if eval_result is None else eval_result["aa"],
            "test_kappa": "" if eval_result is None else eval_result["kappa"],
        })

    if save_best_by_test and best_state is not None:
        model.load_state_dict(best_state)
        final_eval = best_eval
        selection = "best_test"
    else:
        final_eval = evaluate_model(model, test_loader)
        selection = "final_epoch"

    ckpt_path = run_dir / "checkpoints" / DATASET / f"net_params_CrossHL_{tag}_Iter{iteration}.pkl"
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), ckpt_path)
    write_csv(run_dir / "logs" / DATASET / f"epochs_{tag}_Iter{iteration}.csv", epoch_rows)
    np.savetxt(run_dir / "logs" / DATASET / f"confusion_{tag}_Iter{iteration}.csv", final_eval["confusion"], delimiter=",", fmt="%d")

    summary = {
        "dataset": DATASET,
        "experiment": exp_name,
        "prompt_mode": "" if prompt_mode is None else prompt_mode,
        "tag": tag,
        "pct": split_value if split_kind == "pct" else "",
        "shots": split_value if split_kind == "shot" else "",
        "pct_label": split_label(split_kind, split_value),
        "split_label": split_label(split_kind, split_value),
        "iteration": iteration,
        "lambda_sem": lambda_sem,
        "freeze_early": freeze_early,
        "selection": selection,
        "oa": final_eval["oa"],
        "aa": final_eval["aa"],
        "kappa": final_eval["kappa"],
        "seconds": time.time() - start,
        "checkpoint": str(ckpt_path),
    }
    for idx, class_name in enumerate(class_names):
        summary[f"acc_{class_name}"] = final_eval["per_class"][idx]
    append_csv_row(run_dir / "summary.csv", summary)
    print(f"FINAL {tag} Iter{iteration}: OA={summary['oa']:.2f}, AA={summary['aa']:.2f}, Kappa={summary['kappa']:.2f}")
    return summary


# Quick Notebook Sanity Test

This runs one mini optimization step only. It does not create paper results. It is here to prove that the notebook can train directly without using shell commands.


In [ ]:

RUN_MINI_STEP = False

if RUN_MINI_STEP:
    set_seed(SEED)
    mini_model = CrossHL_Transformer(FM=FM, NC=NC, NCLidar=NCLIDAR, Classes=CLASSES, patchsize=PATCH_SIZE).to(DEVICE)
    mini_model.train()
    mini_optimizer = torch.optim.Adam(mini_model.parameters(), lr=5e-4)
    mini_loader = DataLoader(inspect_subset, batch_size=4, shuffle=True)
    mini_hsi, mini_lidar, mini_labels = next(iter(mini_loader))
    mini_hsi = mini_hsi.to(DEVICE)
    mini_lidar = mini_lidar.to(DEVICE)
    mini_labels = mini_labels.to(DEVICE)
    mini_logits = mini_model(mini_hsi, mini_lidar)
    mini_loss = F.cross_entropy(mini_logits, mini_labels)
    mini_optimizer.zero_grad()
    mini_loss.backward()
    mini_optimizer.step()
    print("One notebook training step completed. Loss:", float(mini_loss.detach()))
else:
    print("Mini step skipped. Set RUN_MINI_STEP=True for a one-batch sanity check.")



# Run Cross-Dataset Ablations

Run one dataset or all supported datasets with the same ablation protocol. Semantic experiments require `LOAD_CLIP=True` and the CLIP cell above.


In [ ]:

RUN_ABLATION = False

DATASETS_TO_RUN = ["Trento", "Houston", "MUUFL"]

PCTS = []
SHOTS = [20]
EXPERIMENTS = [
    ("baseline", None, 0.0),
    ("name", "name", 0.01),
    ("spectral", "spectral", 0.01),
    ("spectral_lidar", "spectral_lidar", 0.01),
]

EPOCHS = 200
ITERATIONS = 10
EVAL_INTERVAL = 50
SAVE_BEST_BY_TEST = False
SPECTRAL_NOISE_STD = 0.0
SPECTRAL_GAIN_STD = 0.0
FREEZE_PCT_THRESHOLD = 0.0
SEMANTIC_START_EPOCH = 50
LAMBDA_WARMUP_EPOCHS = 50

FAST_DEV_MODE = False
FAST_TEST_SAMPLES = 1024

completed_run_dirs = []

if RUN_ABLATION:
    needs_clip = any(prompt_mode is not None for _, prompt_mode, _ in EXPERIMENTS)
    for dataset in DATASETS_TO_RUN:
        print("\n" + "#" * 110)
        print(f"DATASET: {dataset}")
        load_dataset_context(dataset)
        if needs_clip:
            ensure_text_prototypes(DATASET)

        run_dir = make_run_dir(name=None)
        completed_run_dirs.append(run_dir)
        print("Run directory:", run_dir)

        if FAST_DEV_MODE:
            test_dataset, fast_counts = make_balanced_eval_subset(test_full, FAST_TEST_SAMPLES, seed=SEED)
            print("FAST_DEV_MODE: balanced test samples limited to", len(test_dataset), "counts", fast_counts)
        else:
            test_dataset = test_full

        test_loader = DataLoader(test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False)
        split_settings = [("pct", pct) for pct in PCTS] + [("shot", shot) for shot in SHOTS]
        for split_kind, split_value in split_settings:
            for iteration in range(ITERATIONS):
                split_seed = SPLIT_SEED + iteration
                if split_kind == "shot":
                    train_subset, counts = make_kshot_subset(train_full, shots=split_value, seed=split_seed)
                else:
                    train_subset, counts = make_fewshot_subset(train_full, pct=split_value, seed=split_seed)
                train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
                print("\n" + "=" * 100)
                print(f"{DATASET} | {split_label(split_kind, split_value)} | iteration {iteration} | samples={len(train_subset)} | counts={counts}")

                for exp_name, prompt_mode, lambda_sem in EXPERIMENTS:
                    train_one_experiment(
                        exp_name=exp_name,
                        prompt_mode=prompt_mode,
                        lambda_sem=lambda_sem,
                        train_loader=train_loader,
                        test_loader=test_loader,
                        run_dir=run_dir,
                        split_kind=split_kind,
                        split_value=split_value,
                        iteration=iteration,
                        epochs=EPOCHS,
                        eval_interval=EVAL_INTERVAL,
                        semantic_start_epoch=SEMANTIC_START_EPOCH,
                        lambda_warmup_epochs=LAMBDA_WARMUP_EPOCHS,
                        save_best_by_test=SAVE_BEST_BY_TEST,
                        spectral_noise_std=SPECTRAL_NOISE_STD,
                        spectral_gain_std=SPECTRAL_GAIN_STD,
                        freeze_pct_threshold=FREEZE_PCT_THRESHOLD,
                    )

        print("Finished dataset summary:", run_dir / "summary.csv")

    print("\nCompleted run directories:")
    for path in completed_run_dirs:
        print(" -", path)
else:
    print("RUN_ABLATION=False. Set True to run the notebook training loop.")
    print("For semantic experiments, set LOAD_CLIP=True and run the CLIP cell first.")


# Optional 1% Stabilization Study

After the main ablation if needed. This tests stronger semantic weight and spectral perturbation.


In [ ]:

RUN_STABILIZATION = False

if RUN_STABILIZATION:
    LOAD_CLIP = True
    DATASETS_TO_RUN = ["Trento", "Houston", "MUUFL"]
    PCTS = []
    SHOTS = [10]
    EXPERIMENTS = [
        ("baseline", None, 0.0),
        ("spectral_lidar", "spectral_lidar", 0.01),
        ("spectral_lidar_high_lambda", "spectral_lidar", 0.03),
    ]
    SPECTRAL_NOISE_STD = 0.01
    SPECTRAL_GAIN_STD = 0.02
    FREEZE_PCT_THRESHOLD = 0.0
    print("Now run the CLIP cell, then the ablation cell above with these settings.")
else:
    print("Stabilization setup not activated.")


# Aggregate Results


In [ ]:
# Example: RESULT_RUN_DIR = PROJECT_ROOT / "runs" / "notebook_Trento_20260526_190000"
RESULT_RUN_DIR = None

if RESULT_RUN_DIR is None:
    runs_dir = PROJECT_ROOT / "runs"
    available = sorted([p for p in runs_dir.glob("*") if (p / "summary.csv").exists()]) if runs_dir.exists() else []
    print("Available runs with summary.csv:")
    for path in available[-10:]:
        print(" -", path)
    if available:
        RESULT_RUN_DIR = available[-1]
        print("Using latest:", RESULT_RUN_DIR)

if RESULT_RUN_DIR is not None and (RESULT_RUN_DIR / "summary.csv").exists():
    summary_df = pd.read_csv(RESULT_RUN_DIR / "summary.csv")
    aggregate_df = summary_df.groupby(["pct_label", "experiment"], as_index=False).agg(
        oa_mean=("oa", "mean"),
        oa_std=("oa", "std"),
        aa_mean=("aa", "mean"),
        aa_std=("aa", "std"),
        kappa_mean=("kappa", "mean"),
        kappa_std=("kappa", "std"),
        n=("oa", "count"),
    )
    display(aggregate_df)
    out_path = RESULT_RUN_DIR / "analysis" / "aggregate_summary.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    aggregate_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    base_df = summary_df[summary_df["experiment"] == "baseline"][["pct_label", "iteration", "oa", "aa", "kappa"]]
    base_df = base_df.rename(columns={"oa": "baseline_oa", "aa": "baseline_aa", "kappa": "baseline_kappa"})
    paired_df = summary_df.merge(base_df, on=["pct_label", "iteration"], how="left")
    paired_df["delta_oa"] = paired_df["oa"] - paired_df["baseline_oa"]
    paired_df["delta_aa"] = paired_df["aa"] - paired_df["baseline_aa"]
    paired_df["delta_kappa"] = paired_df["kappa"] - paired_df["baseline_kappa"]
    delta_summary_df = paired_df.groupby(["pct_label", "experiment"], as_index=False).agg(
        delta_oa_mean=("delta_oa", "mean"),
        delta_oa_std=("delta_oa", "std"),
        delta_aa_mean=("delta_aa", "mean"),
        delta_kappa_mean=("delta_kappa", "mean"),
        n=("delta_oa", "count"),
    )
    display(delta_summary_df)
    paired_path = RESULT_RUN_DIR / "analysis" / "paired_deltas.csv"
    delta_path = RESULT_RUN_DIR / "analysis" / "paired_delta_summary.csv"
    paired_df.to_csv(paired_path, index=False)
    delta_summary_df.to_csv(delta_path, index=False)
    print("Saved:", paired_path)
    print("Saved:", delta_path)
else:
    print("No completed run selected yet.")


# Plot OA Trends


In [ ]:

if RESULT_RUN_DIR is not None and (RESULT_RUN_DIR / "summary.csv").exists():
    summary_df = pd.read_csv(RESULT_RUN_DIR / "summary.csv")
    aggregate_df = summary_df.groupby(["pct_label", "experiment"], as_index=False).agg(
        oa_mean=("oa", "mean"),
        oa_std=("oa", "std"),
    )
    experiments = ["baseline", "name", "spectral", "spectral_lidar", "spectral_lidar_high_lambda"]
    styles = {
        "baseline": ("#d62728", "o", "-"),
        "name": ("#7f7f7f", "D", ":"),
        "spectral": ("#1f77b4", "s", "--"),
        "spectral_lidar": ("#2ca02c", "^", "-."),
        "spectral_lidar_high_lambda": ("#9467bd", "P", "-"),
    }
    present_pcts = sorted(set(aggregate_df["pct_label"]), key=split_sort_key)
    x = np.arange(len(present_pcts))
    plt.figure(figsize=(9, 6))
    for exp in experiments:
        rows = aggregate_df[aggregate_df["experiment"] == exp]
        if rows.empty:
            continue
        means = []
        stds = []
        for pct in present_pcts:
            row = rows[rows["pct_label"] == pct]
            means.append(np.nan if row.empty else float(row["oa_mean"].iloc[0]))
            stds.append(0.0 if row.empty or pd.isna(row["oa_std"].iloc[0]) else float(row["oa_std"].iloc[0]))
        color, marker, linestyle = styles[exp]
        plt.errorbar(x, means, yerr=stds, label=exp, color=color, marker=marker, linestyle=linestyle, capsize=5)
    plt.xticks(x, present_pcts)
    plt.xlabel("Few-shot setting")
    plt.ylabel("Overall accuracy (%)")
    plt.title("CrossHL-VLM ablation")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    fig_path = RESULT_RUN_DIR / "analysis" / "oa_trends.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print("Saved:", fig_path)
else:
    print("No completed run selected yet.")


# t-SNE Feature Analysis


In [ ]:

def load_checkpoint_model(checkpoint_path):
    model = CrossHL_Transformer(FM=FM, NC=NC, NCLidar=NCLIDAR, Classes=CLASSES, patchsize=PATCH_SIZE).to(DEVICE)
    state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model


def extract_cls_features(model, dataset, batch_size=500):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features = []
    labels = []
    with torch.no_grad():
        for hsi_batch, lidar_batch, label_batch in loader:
            hsi_batch = hsi_batch.to(DEVICE)
            lidar_batch = lidar_batch.to(DEVICE)
            _, cls = model(hsi_batch, lidar_batch, return_features=True)
            features.append(cls.cpu().numpy())
            labels.append(label_batch.numpy())
    return np.concatenate(features), np.concatenate(labels)


def plot_tsne_for_run(run_dir, pct_label="20-shot", iteration=0, num_samples=1500):
    summary_df = pd.read_csv(run_dir / "summary.csv")
    dataset = summary_df["dataset"].iloc[0]
    load_dataset_context(dataset)
    experiments = ["baseline", "name", "spectral", "spectral_lidar"]
    fig, axes = plt.subplots(1, len(experiments), figsize=(6 * len(experiments), 5))
    sample_size = min(num_samples, len(test_full))
    sample_idx = np.random.default_rng(42).choice(len(test_full), sample_size, replace=False)

    for ax, exp in zip(axes, experiments):
        row = summary_df[(summary_df["pct_label"] == pct_label) & (summary_df["iteration"] == iteration) & (summary_df["experiment"] == exp)]
        if row.empty:
            ax.set_title(f"Missing {exp}")
            ax.axis("off")
            continue
        checkpoint = Path(row["checkpoint"].iloc[0])
        model = load_checkpoint_model(checkpoint)
        features, labels = extract_cls_features(model, test_full)
        embedded = TSNE(n_components=2, random_state=42, init="pca", perplexity=min(30, max(5, sample_size // 20))).fit_transform(features[sample_idx])
        sampled_labels = labels[sample_idx]
        for class_id, class_name in enumerate(class_names):
            mask = sampled_labels == class_id
            ax.scatter(embedded[mask, 0], embedded[mask, 1], s=16, alpha=0.65, label=class_name)
        ax.set_title(exp)
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(len(class_names), 6))
    fig.suptitle(f"{dataset} CLS feature t-SNE, {pct_label}, iteration {iteration}")
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    safe_label = pct_label.replace("%", "pct").replace(" ", "_")
    fig_path = run_dir / "analysis" / f"tsne_cls_{safe_label}_iter{iteration}.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print("Saved:", fig_path)

RUN_TSNE = False
if RUN_TSNE and RESULT_RUN_DIR is not None:
    plot_tsne_for_run(RESULT_RUN_DIR, pct_label="20-shot", iteration=0)
else:
    print("Set RUN_TSNE=True after a completed run.")


# Legacy Result Tables

Load archived result tables when available.


In [ ]:
legacy_dir = PROJECT_ROOT / "results" / "legacy"
for filename in ["Master_Paper_Results.csv", "Master_PerClass_Accuracy.csv"]:
    path = legacy_dir / filename
    if path.exists():
        print("\n" + filename)
        display(pd.read_csv(path))
    else:
        print("Missing:", path)
